# Bending vs LTB: Predicted and Actual Failure Strengths

For each beam in the 2-variable dataset, compute:
- Predicted bending failure load (from Ix)
- Predicted LTB failure load (from Iy, J)
- Stability ratio R = Mcr/My
- Governing predicted load = min(bending, LTB)
- Compare to actual measured failure load

Material: Sunlu PLA+ 2.0 (blue)  
Datasheet: E_flex = 2740 MPa, sigma_flex = 81.8 MPa, density = 1.21 g/cm³

No fillets in these designs (sharp web-flange junctions).

J is computed two ways:
1. Thin-wall approximation: J = (H*b^3 + 2*B*h^3)/3. Overestimates J for thick elements.
2. Timoshenko correction: J = sum(beta_i * a_i * t_i^3) where beta accounts for finite thickness.

The thin-wall formula overestimates J by 5-44% across this dataset, which overestimates Mcr by 3-20%. This is unconservative for LTB prediction.

In [ ]:
import numpy as np
import pandas as pd

# --- Sunlu PLA+ 2.0 datasheet (ASTM D790 / D638) ---
E = 2.74e9           # Pa, flexural modulus
SIGMA_Y = 81.8e6     # Pa, flexural strength
DENSITY = 1210       # kg/m^3
G = E / 2.6          # shear modulus, assuming nu ~ 0.3

# --- Geometry constants ---
TOTAL_H = 25.0       # mm, total beam height
B = 16.0             # mm, flange width
L = 0.2023           # m, span (3-point bending)
C1 = 1.35            # moment gradient factor for 3-point bending

In [ ]:
def calc_Ix(H_mm, b_mm, B_mm=B, total_h=TOTAL_H):
    """Strong-axis second moment of area (m^4)."""
    H = H_mm / 1e3
    b = b_mm / 1e3
    Bf = B_mm / 1e3
    h = (total_h / 1e3 - H) / 2.0
    if h <= 0:
        return 0.0
    return (b * H**3) / 12 + 2 * (Bf * h**3 / 12 + Bf * h * ((H + h) / 2)**2)

def calc_Iy(H_mm, b_mm, B_mm=B, total_h=TOTAL_H):
    """Weak-axis second moment of area (m^4)."""
    H = H_mm / 1e3
    b = b_mm / 1e3
    Bf = B_mm / 1e3
    h = (total_h / 1e3 - H) / 2.0
    if h <= 0:
        return 0.0
    return (H * b**3) / 12 + 2 * (h * Bf**3) / 12

def _beta(t, a):
    """Timoshenko correction for torsional constant of a rectangle.
    t = short dimension, a = long dimension. Returns beta such that
    J_rect = beta * a * t^3.  Thin-wall limit: beta -> 1/3."""
    if a <= 0 or t <= 0:
        return 1.0 / 3.0
    r = min(t, a) / max(t, a)
    return (1.0 / 3.0) * (1.0 - 0.63 * r + 0.052 * r**5)

def calc_J_thinwall(H_mm, b_mm, B_mm=B, total_h=TOTAL_H):
    """Torsional constant, thin-wall approximation (m^4). Overestimates J."""
    H, b, Bf = H_mm/1e3, b_mm/1e3, B_mm/1e3
    h = (total_h/1e3 - H) / 2.0
    if h <= 0:
        return 0.0
    return (H * b**3 + 2 * Bf * h**3) / 3

def calc_J_exact(H_mm, b_mm, B_mm=B, total_h=TOTAL_H):
    """Torsional constant with Timoshenko finite-thickness correction (m^4)."""
    H, b, Bf = H_mm/1e3, b_mm/1e3, B_mm/1e3
    h = (total_h/1e3 - H) / 2.0
    if h <= 0:
        return 0.0
    J_web = _beta(min(b, H), max(b, H)) * max(b, H) * min(b, H)**3
    J_fl  = _beta(min(h, Bf), max(h, Bf)) * max(h, Bf) * min(h, Bf)**3
    return J_web + 2 * J_fl

def beam_calcs(H_mm, b_mm):
    h_mm = (TOTAL_H - H_mm) / 2.0
    Ix = calc_Ix(H_mm, b_mm)
    Iy = calc_Iy(H_mm, b_mm)
    J_tw = calc_J_thinwall(H_mm, b_mm)
    J_ex = calc_J_exact(H_mm, b_mm)
    y_max = TOTAL_H / 2e3

    My = SIGMA_Y * Ix / y_max

    def _mcr(J):
        return (C1 * np.pi / L) * np.sqrt(E * Iy * G * J) if (Iy > 0 and J > 0) else 0.0

    Mcr_tw = _mcr(J_tw)
    Mcr_ex = _mcr(J_ex)
    R_tw = Mcr_tw / My if My > 0 else 0.0
    R_ex = Mcr_ex / My if My > 0 else 0.0

    P_bend = 4 * My / L
    P_ltb_tw = 4 * Mcr_tw / L
    P_ltb_ex = 4 * Mcr_ex / L
    P_gov_tw = min(P_bend, P_ltb_tw)
    P_gov_ex = min(P_bend, P_ltb_ex)

    return {
        'h_mm': h_mm,
        'R_thinwall': R_tw,
        'R_exact': R_ex,
        'P_bend_N': P_bend,
        'P_ltb_tw_N': P_ltb_tw,
        'P_ltb_ex_N': P_ltb_ex,
        'P_gov_tw_N': P_gov_tw,
        'P_gov_ex_N': P_gov_ex,
        'J_err_pct': (J_tw - J_ex) / J_ex * 100 if J_ex > 0 else 0,
    }

In [ ]:
df = pd.read_csv('../data/I_beam_data_2var.csv')

rows = []
for _, r in df.iterrows():
    b = r['b_web_mm']
    H = r['H_web_mm']
    actual = r['Strength N']
    c = beam_calcs(H, b)
    rows.append({
        'Beam': r['Beam Number'],
        'b': b,
        'H': H,
        'R_tw': c['R_thinwall'],
        'R_ex': c['R_exact'],
        'P_bend': c['P_bend_N'],
        'P_ltb_tw': c['P_ltb_tw_N'],
        'P_ltb_ex': c['P_ltb_ex_N'],
        'P_gov_tw': c['P_gov_tw_N'],
        'P_gov_ex': c['P_gov_ex_N'],
        'P_actual': actual,
        'Act/Pred_tw': actual / c['P_gov_tw_N'] if c['P_gov_tw_N'] > 0 else 0,
        'Act/Pred_ex': actual / c['P_gov_ex_N'] if c['P_gov_ex_N'] > 0 else 0,
        'J_err%': c['J_err_pct'],
        'Failure mode': str(r.get('Column 1', '')),
        'Notes': str(r.get('Column 2', '')),
    })

result = pd.DataFrame(rows)

fmt = {
    'b': '{:.1f}', 'H': '{:.1f}',
    'R_tw': '{:.2f}', 'R_ex': '{:.2f}',
    'P_bend': '{:.0f}', 'P_ltb_tw': '{:.0f}', 'P_ltb_ex': '{:.0f}',
    'P_gov_tw': '{:.0f}', 'P_gov_ex': '{:.0f}', 'P_actual': '{:.0f}',
    'Act/Pred_tw': '{:.2f}', 'Act/Pred_ex': '{:.2f}', 'J_err%': '{:.0f}',
}

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 300)
result.style.format(fmt)

In [ ]:
print('=== Thin-wall vs Timoshenko J comparison ===')
print(f'Mean J overestimate (thin-wall): {result["J_err%"].mean():.1f}%')
print(f'Max  J overestimate (thin-wall): {result["J_err%"].max():.1f}%')
print()
print('=== Prediction accuracy: thin-wall J ===')
print(f'Mean Act/Pred: {result["Act/Pred_tw"].mean():.2f}  Std: {result["Act/Pred_tw"].std():.2f}')
print()
print('=== Prediction accuracy: Timoshenko J ===')
print(f'Mean Act/Pred: {result["Act/Pred_ex"].mean():.2f}  Std: {result["Act/Pred_ex"].std():.2f}')
print()

print('Beams where LTB governs (R_exact < 1):')
ltb_gov = result[result['R_ex'] < 1.0]
if len(ltb_gov) > 0:
    print(ltb_gov[['Beam', 'b', 'H', 'R_ex', 'P_ltb_ex', 'P_actual', 'Act/Pred_ex', 'Failure mode']].to_string(index=False))

print()
print('Beams where bending governs (R_exact >= 1):')
bend_gov = result[result['R_ex'] >= 1.0]
if len(bend_gov) > 0:
    print(bend_gov[['Beam', 'b', 'H', 'R_ex', 'P_bend', 'P_actual', 'Act/Pred_ex', 'Failure mode']].to_string(index=False))